# Learning Rate Schedule

**Llama 3 Pre-training:** "We pre-train **Llama 3 405B** using **AdamW** with a **peak learning rate of** $\mathbf{8 × 10^{−5}}$ , **a linear warm up of 8,000 steps**, and **a cosine learning rate schedule decaying to** $\mathbf{8 × 10^{−7}}$ **over** $\mathbf{1{,}200{,}000}$ **steps**. We use a lower batch size early in training to improve training stability, and increase it subsequently to improve efficiency. Specifically, we use an initial batch size of 4M tokens and sequences of length $4{,}096$, and double these values to a batch size of 8M sequences of $8{,}192$ tokens after pre-training 252M tokens. We double the batch size again to 16M after pre-training on $2.87T$ tokens. We found this training recipe to be very stable: we observed few loss spikes and did not require interventions to correct for model training divergence." 

**Llama 2 paper:**
- Hyperparameters. We trained using the **AdamW** optimizer (Loshchilov and Hutter, 2017), with $\boldsymbol{\beta_1 = 0.9}$, $\boldsymbol{\beta_2 = 0.95}$, $\cancel{eps = 10^{−5}}$. ~~We use a cosine learning rate schedule~~, ~~with warmup of~~ $\cancel{2000}$ ~~steps~~, and **decay final learning rate down to 10% of the peak learning rate**. We use a **weight decay of** $\boldsymbol{0.1}$ and ~~gradient clipping of 1.0~~. Figure 5 (a) shows the training loss for Llama 2 with these hyperparameters.

**Note:** In the Llama 3 paper, specifically when they are pre-training the encoders for the **multi-modal Vision** they mentioned "We use a global batch size of $16{,}384$ and a cosine learning rate schedule with initial learning rate $10 × 10^{−4}$ and a weight decay of $0.1$. The initial learning rate was determined based on small-scale experiments."
#TODO is weight decay 0.1 or 0.01? here: weight decay of $0.1$???

In [1]:
import EasyJupyter
import torch
import torch.nn as nn

In [2]:
import math
import torch.optim as optim
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig
    from model.decoder import Decoder

In [3]:
def get_lr_multiplier(cfg, current_step: int):
    """
    Calculates the learning rate multiplier for the Llama 3 schedule.
    1. Linear warmup from 0 to peak_lr.
    2. Linear decay from peak_lr down to 1% or 10% of peak_lr.
    """
    anneal_start = cfg.pre_training_stages_steps["annealing_perc"]
    min_lr_ratio = getattr(cfg, "min_lr_ratio", 0.1)

    # Annealing sub-stage, linear decay from min_lr_ratio to 0.
    if current_step >= anneal_start:
        annealing_steps = cfg.total_training_steps - anneal_start
        steps_into_annealing = current_step - anneal_start
        # linear decay the remaining multiplier to 0
        decay_factor = 1.0 - (steps_into_annealing / max(1, annealing_steps))
        return min_lr_ratio * max(0.0, decay_factor)

    # Linear warmup phase in the initial sub-stage
    if current_step < cfg.warmup_steps:
        # return a fraction from near 0.0 up to 1.0
        return float(current_step) / float(max(1, cfg.warmup_steps))

    # Cosine Decay Phase
    # Calculate progress from 0.0 (end of warmup) to 1.0 (end of token budget)
    decay_progress = float(current_step - cfg.warmup_steps) / float(
        max(1, cfg.total_training_steps - cfg.warmup_steps)
    )

    # Cap the step so the learning rate doesn't start rising again
    decay_progress = min(1.0, decay_progress)

    # Cosine: scales down from 1.0 to 0.0
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * decay_progress))

    # Scale it to the stop at min_lr_ratio (10% or 1% depending on config) instead of hitting 0
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay


def get_scheduler(
    cfg: BaseConfig, optimizer: torch.optim.AdamW, isAnnealing: bool = False
):
    """
    The scheduler will be different depending on the training phase configurations from cfg.

    Args:
        isAnnealing: True if we are at the annealing stage of pre-training, where we decayed the learning rate to $0$ past the `min_lr_ratio`, change the data mix to the absolute highest quality math and reasoning data sources.
    """
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer=optimizer,
        lr_lambda=lambda step: get_lr_multiplier(
            current_step=step,
            cfg=cfg,
        ),
    )


def get_optimizer(cfg: BaseConfig, model: nn.Module):
    """ """
    return optim.AdamW(
        model.parameters(),
        lr=cfg.peak_lr,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,  # TODO verify params in paper
    )

## Test

In [4]:
# @i-c
# Test: LR Schedule Example
print("\n------ LR Schedule Example For The Llama 3 405B Model ------\n")
# Dummy Hyperparameters
from llama_configs import Llama3_scaled_down

cfg = Llama3_scaled_down()
cfg.warmup_steps= 8_000
cfg.peak_lr = 8e-5
cfg.token_budget = 1_200_000 * cfg.global_batch_size_tokens

print(f"Total training steps: {cfg.total_training_steps:,}")
print(f"Warmup ends at step: {cfg.warmup_steps}")
print(
    f"Peak learning rate: {cfg.peak_lr:.8f} | Min learning rate: {cfg.peak_lr * cfg.min_lr_ratio:.8f}"
)
print(f"\n\n{'Step':>10} | {'Learning-Rate':>15} | {'Phase'}")
print("-" * 55)

milestones = [  # Training milestones
    1,          # Start warmup
    4_000,      # Halfway through warmup
    8_000,      # Peak LR (end of warmup)
    8_001,      # Start of decay (The learning rate immediately starts to drop)
    100_000,    # Early decay
    306_000,    # 25% through decay
    604_000,    # 50% through decay
    902_000,    # 75% through decay
    1_200_000,  # End of decay, hits 1% of peak learning rate
    1_250_000,  # Past total steps (stays flat at 1%)
]

for step in milestones:
    # Get multiplier (0.0 to 1.0)
    multiplier = get_lr_multiplier(
        current_step=step, cfg=cfg
    )

    # Apply it to the peak learning rate
    lr = cfg.peak_lr * multiplier

    if step < cfg.warmup_steps:
        phase = "Warmup (Linear up ↑)"
    elif step == cfg.warmup_steps:
        phase = "Peak LR"
    elif step < cfg.total_training_steps:
        phase = "Decay (Cosine down ↓)"
    else:
        phase = "Minimum LR (Flat →)"

    print(f"{step:10d} | {lr:15.8f} | {phase}")


------ LR Schedule Example For The Llama 3 405B Model ------


Project Root: /Users/tonyavis/Main/How_to_build_an_LLM
Total training steps: 1,200,000
Warmup ends at step: 8000
Peak learning rate: 0.00008000 | Min learning rate: 0.00000800


      Step |   Learning-Rate | Phase
-------------------------------------------------------
         1 |      0.00000800 | Warmup (Linear up ↑)
      4000 |      0.00000797 | Warmup (Linear up ↑)
      8000 |      0.00000795 | Peak LR
      8001 |      0.00000795 | Decay (Cosine down ↓)
    100000 |      0.00000733 | Decay (Cosine down ↓)
    306000 |      0.00000596 | Decay (Cosine down ↓)
    604000 |      0.00000397 | Decay (Cosine down ↓)
    902000 |      0.00000199 | Decay (Cosine down ↓)
   1200000 |      0.00000000 | Minimum LR (Flat →)
   1250000 |      0.00000000 | Minimum LR (Flat →)
